# Pacotes

In [1]:
import pandas as pd

from sklearn import  model_selection
from sklearn.tree import DecisionTreeClassifier
from scipy import stats
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.metrics import confusion_matrix

from sklearn import tree
from matplotlib import pyplot as plt

from sklearn.model_selection import KFold 
from sklearn.model_selection import cross_val_score 

import numpy as np
import seaborn as sns
from prettytable import PrettyTable
import sklearn

# Modelo

In [32]:
for i in range(1,18):
    
    print('')
    print('#_______________ Arquivo ', i, '________________#')
    print('')
    
    i=str(i)

    
    # Extração dos dados
    
    X_treino = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_ins_c_over.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    X_treino = X_treino[(X_treino.loc[:,'flg_comunidades' ] == 1)]
    
    print('')
    print('Tamanho do treino e quantidade de ocorrências')
    print(len(X_treino.index))
    print(round(sum(X_treino['flg_ocorrencia'])))
    print('')
    
    X_teste = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_oos.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    X_teste = X_teste[(X_teste.loc[:,'flg_comunidades' ] == 1)]
    
    print('')
    print('Tamanho do teste e quantidade de ocorrências')
    print(len(X_teste.index))
    print(round(sum(X_teste['flg_ocorrencia'])))
    print('')
    
    oot = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_final_oot.csv'
                     ,sep = ','
                     ,decimal='.'
                      )
    
    oot = oot[(oot.loc[:,'flg_comunidades' ] == 1)]
    
    print('')
    print('Tamanho do teste 2 meses a frente e quantidade de ocorrências')
    print(len(oot.index))
    print(round(sum(oot['flg_ocorrencia'])))
    print('')
    
    if round(sum(X_treino['flg_ocorrencia'])) == 0 :
        continue
        
    if round(sum(X_teste['flg_ocorrencia'])) == 0 :
        continue
    
    if round(sum(oot['flg_ocorrencia'])) == 0 :
        continue
    
    #Ajuste na flg_ocorrencia
    
    y_treino = X_treino['flg_ocorrencia']
    X_treino = X_treino.drop('flg_ocorrencia', axis=1)
    n = len(X_treino.columns)
    
    y_teste = X_teste['flg_ocorrencia']
    X_teste = X_teste.drop('flg_ocorrencia', axis=1)
    
    ###########################################
    ############ Árvore de Decisão ############
    ###########################################
    
    maxi = int(sum(y_treino)-1) if sum(y_treino) < 10 else 10
    min_s = int((sum(y_treino)/10)+1) if sum(y_treino) < 300 else 30
    
    classificador_arvore = DecisionTreeClassifier()

    lista_parametros = {'criterion': ['gini', 'entropy', 'log_loss'],
        'splitter': ['best', 'random'],
        'max_depth': [i + 1 for i in range(2,30,1)],
        'min_samples_split': [i + 1 for i in range(1,int(maxi-1),1)],
        'min_impurity_decrease': stats.uniform(0, 1)
        }

    rand_search = RandomizedSearchCV(classificador_arvore, 
                                     param_distributions = lista_parametros,
                                     cv = int(maxi-1),
                                     random_state = 42,
                                     scoring = 'f1') 
    rand_search.fit(X_treino.iloc[:,3:n], y_treino) 
    
    print('')
    print(rand_search.best_estimator_)
    print('')
    
    classificador_arvore = rand_search.best_estimator_
    classificador_arvore.fit(X_treino.iloc[:,3:n], y_treino) 
    
    print('')
    text_representation = tree.export_text(classificador_arvore, feature_names = list(X_treino.iloc[:,3:n].columns) )
    print(text_representation)
    print('')
    
    #############################################
    ############ Qualidade do modelo ############
    #############################################
    
    kfold  = KFold(n_splits=maxi, shuffle=False)
    result_arvore = cross_val_score(classificador_arvore, X_treino.iloc[:,3:n], y_treino, cv = kfold)
    #K-fold
    #'Média da acurácia:'
    print(np.mean(result_arvore))
    #'Variância da acurácia'
    print(np.std(result_arvore))

    #Teste - OOS
    y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])),2))

    #Teste - OOT
    y_teste, classificador_arvore.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(oot.iloc[:,0], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(oot.iloc[:,0], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(oot.iloc[:,0], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(oot.iloc[:,0], classificador_arvore.predict(oot.iloc[:,4:n+1])),2))


#_______________ Arquivo  1 ________________#


Tamanho do treino e quantidade de ocorrências
3354
1677


Tamanho do teste e quantidade de ocorrências
894
11


Tamanho do teste 2 meses a frente e quantidade de ocorrências
109805
186


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- chuva_d5_vintequatrohoras <= 0.10
|   |--- Declividade_numerica <= 18.40
|   |   |--- class: 0.0
|   |--- Declividade_numerica >  18.40
|   |   |--- Curv_Vertical_5classes <= 1.00
|   |   |   |--- Altitude_numerica <= 40.70
|   |   |   |   |--- mes_ocorrencia <= 6.50
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- mes_ocorrencia >  6.50
|   |   |   |   |   |--- Declividade_numerica <= 21.42
|   |   |   |   |   |   |--- chuva_d4_setentaduashoras <= 0.70
|   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |   |--- chuva_d4_setentaduashoras >  0.70
|   |   |   |  

0.9400820699558684
0.03055415929340252
0.92
0.03
0.44
0.06
0.41
0.0
0.5
0.0

#_______________ Arquivo  3 ________________#


Tamanho do treino e quantidade de ocorrências
48024
24012


Tamanho do teste e quantidade de ocorrências
11939
76


Tamanho do teste 2 meses a frente e quantidade de ocorrências
2025
4


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Forma_de_terreno_classes <= 1.00
|   |--- Declividade_numerica <= 13.82
|   |   |--- Declividade_numerica <= 8.82
|   |   |   |--- Relevo_sombreado_numerico <= 0.56
|   |   |   |   |--- class: 1.0
|   |   |   |--- Relevo_sombreado_numerico >  0.56
|   |   |   |   |--- class: 0.0
|   |   |--- Declividade_numerica >  8.82
|   |   |   |--- Relevo_sombreado_numerico <= 0.45
|   |   |   |   |--- class: 0.0
|   |   |   |--- Relevo_sombreado_numerico >  0.45
|   |   |   |   |--- class: 1.0
|   |--- Declividade_num

0.930807358444149
0.02589413156219985
0.92
0.03
0.42
0.06
0.61
0.0
0.25
0.0

#_______________ Arquivo  4 ________________#


Tamanho do treino e quantidade de ocorrências
49152
24576


Tamanho do teste e quantidade de ocorrências
12226
76


Tamanho do teste 2 meses a frente e quantidade de ocorrências
504
1


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Forma_de_terreno_classes <= 1.00
|   |--- Declividade_numerica <= 13.89
|   |   |--- Declividade_numerica <= 8.82
|   |   |   |--- Area_Urbana <= 0.50
|   |   |   |   |--- class: 1.0
|   |   |   |--- Area_Urbana >  0.50
|   |   |   |   |--- class: 0.0
|   |   |--- Declividade_numerica >  8.82
|   |   |   |--- Altitude_numerica <= 20.62
|   |   |   |   |--- class: 0.0
|   |   |   |--- Altitude_numerica >  20.62
|   |   |   |   |--- class: 1.0
|   |--- Declividade_numerica >  13.89
|   |   |--- Relevo_sombread

0.9287124153738038
0.03584430997491167
0.9
0.02
0.41
0.05
1.0
0.0
0.0
0.0

#_______________ Arquivo  5 ________________#



C:\Users\debor\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

0.8956470021540481
0.07254899105758712
0.82
0.01
0.41
0.01
0.84
0.01
0.13
0.01

#_______________ Arquivo  12 ________________#


Tamanho do treino e quantidade de ocorrências
249148
124574


Tamanho do teste e quantidade de ocorrências
62418
149


Tamanho do teste 2 meses a frente e quantidade de ocorrências
3930
40


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Curv_Vertical_5classes <= 5.00
|   |--- Curv_Vertical_3classes <= 1.00
|   |   |--- Forma_de_terreno_classes <= 1.00
|   |   |   |--- noventaseishoras <= 36.28
|   |   |   |   |--- Declividade_numerica <= 20.15
|   |   |   |   |   |--- chuva_d4_setentaduashoras <= 0.60
|   |   |   |   |   |   |--- chuva_d4_vintequatrohoras <= 0.00
|   |   |   |   |   |   |   |--- ADD_divisores_talvegues <= 13.16
|   |   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |   |   |--- ADD_divisores_talvegues 

0.8645670427563212
0.06532624062192363
0.84
0.0
0.32
0.01
0.91
0.01
0.05
0.01

#_______________ Arquivo  13 ________________#


Tamanho do treino e quantidade de ocorrências
280438
140219


Tamanho do teste e quantidade de ocorrências
70179
164


Tamanho do teste 2 meses a frente e quantidade de ocorrências
0
0


#_______________ Arquivo  14 ________________#


Tamanho do treino e quantidade de ocorrências
281162
140581


Tamanho do teste e quantidade de ocorrências
70372
164


Tamanho do teste 2 meses a frente e quantidade de ocorrências
759
11


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Curv_Vertical_5classes <= 5.00
|   |--- Curv_Vertical_3classes <= 1.00
|   |   |--- Forma_de_terreno_classes <= 1.00
|   |   |   |--- noventaseishoras <= 27.82
|   |   |   |   |--- Declividade_numerica <= 20.15
|   |   |   |   |   |--- chuva_d4_setentaduashoras <= 0.60


0.8568408173014703
0.036330749379579204
0.85
0.01
0.34
0.01
0.87
0.0
0.0
0.0

#_______________ Arquivo  15 ________________#


Tamanho do treino e quantidade de ocorrências
289342
144671


Tamanho do teste e quantidade de ocorrências
72390
167


Tamanho do teste 2 meses a frente e quantidade de ocorrências
0
0


#_______________ Arquivo  16 ________________#


Tamanho do treino e quantidade de ocorrências
289342
144671


Tamanho do teste e quantidade de ocorrências
72390
167


Tamanho do teste 2 meses a frente e quantidade de ocorrências
68
1


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Curv_Vertical_5classes <= 5.00
|   |--- Curv_Vertical_3classes <= 1.00
|   |   |--- mes_ocorrencia <= 4.50
|   |   |   |--- chuva_d4_mes <= 139.60
|   |   |   |   |--- chuva_d4_mes <= 52.80
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- chuva_d4_mes >  52.80
|   

0.8645206031844352
0.058289635388309884
0.83
0.0
0.37
0.01
0.01
0.01
1.0
0.03

#_______________ Arquivo  17 ________________#


Tamanho do treino e quantidade de ocorrências
290134
145067


Tamanho do teste e quantidade de ocorrências
72561
168


Tamanho do teste 2 meses a frente e quantidade de ocorrências
348
6


DecisionTreeClassifier(criterion='entropy', max_depth=8,
                       min_impurity_decrease=0.0007787658410143283,
                       min_samples_split=5)


|--- Curv_Vertical_5classes <= 5.00
|   |--- Curv_Vertical_3classes <= 1.00
|   |   |--- mes_ocorrencia <= 4.50
|   |   |   |--- chuva_d4_mes <= 140.20
|   |   |   |   |--- chuva_d4_mes <= 52.80
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- chuva_d4_mes >  52.80
|   |   |   |   |   |--- mes_ocorrencia <= 1.50
|   |   |   |   |   |   |--- chuva_d4_quarentaoitohoras <= 0.61
|   |   |   |   |   |   |   |--- chuva_d4_mes <= 78.97
|   |   |   |   |   |   |   |   |--- class: 1.0
|   |   |   |   |   |  

0.8696231331929841
0.058853169061418555
0.83
0.01
0.38
0.01
0.69
0.01
0.17
0.02
